<center>
<img src="../images/logo-poster-transparent.png" width="400px" style="opacity:0.7">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



# DeepAgents as a Harness

Notebook 01 covered the Deep Agents core — planning, a virtual filesystem, subagents, and human-in-the-loop. This notebook adds the remaining harness knobs, each as **one parameter** of `create_deep_agent`:

| Harness concern | Parameter |
|---|---|
| Skills (on-demand procedures) | `skills=[...]` |
| Always-loaded memory / context | `memory=["/memory/AGENTS.md"]` |
| Tool-call policy (allow / deny / ask) | `permissions=[...]` |
| Retrying failed tool calls | `ToolRetryMiddleware` |
| Structured output | `response_format=Schema` |

This notebook demonstrates each in isolation, with the same model throughout.


In [2]:
import os

from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
print("OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "NOT SET — fix .env first")

# Book default. gpt-5 models reject non-default temperature, so none is set.
model = ChatOpenAI(model="gpt-5-mini", timeout=300.0)


OPENAI_API_KEY: set


## 1. Skills: procedures the agent loads on demand

A deepagents **skill** is a folder with a `SKILL.md` whose YAML frontmatter carries a `name` and a `description`. The harness shows the model only the name+description catalog (level 1); the body is read **only when the skill is used** — progressive disclosure, so a large library costs almost no context.

With the default `StateBackend`, skill files live in the agent's virtual filesystem, so we can hand them over in the `invoke` call itself via `files={...}` — no disk setup at all. Each value has to be a file record (`create_file_data(text)`), not a bare string: the backend reads `file["content"]`.


In [3]:
HAIKU_SKILL = """---
name: haiku-report
description: Write a status report entirely in haiku form. Use whenever the user asks for a report, status update, or summary "as haiku" or "in haiku form".
---

# Haiku Report

1. Identify the 2-4 key facts to report.
2. Write one haiku (5-7-5 syllables) per fact. Count the syllables.
3. End with a single-line prose postscript: "(haiku report complete)".
4. Never pad with facts that were not given.
"""

skilled_agent = create_deep_agent(
    model=model,
    system_prompt="You are a concise assistant.",
    skills=["/skills/"],          # catalog of every skill folder under this path
)

skill_files = {"/skills/haiku-report/SKILL.md": create_file_data(HAIKU_SKILL)}

# Control run: normal question — the skill is NOT relevant, so SKILL.md is never read
result_plain = skilled_agent.invoke({
    "messages": [{"role": "user", "content": "What is 2 + 2?"}],
    "files": skill_files,
})

# Skill run: asks for haiku form — the agent reads SKILL.md on demand before answering
result = skilled_agent.invoke({
    "messages": [{"role": "user", "content":
        "Status: the data pipeline shipped on Monday; costs fell 12%. Report this in haiku form."}],
    "files": skill_files,
})
print(result["messages"][-1].content)


Pipeline launched Monday
Data flows now live, steady
Teams breathe, tasks move on

Costs dropped twelve percent
Savings ease the budget strain
Growth path opens wide

(haiku report complete)


Compare the intermediate messages across the two runs: on the plain question, the skill cost zero `read_file` calls; on the haiku request, the model saw the skill *catalog* in its system prompt and `read_file`'d the `SKILL.md` body before following it — load-on-use.


In [4]:
def show_tool_trace(res, label: str) -> None:
    print(f"=== {label} ===")
    any_tools = False
    for m in res["messages"]:
        kind = type(m).__name__
        if kind == "ToolMessage":
            print(f"  [tool result] {str(m.content)[:90]}")
            any_tools = True
        elif getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  [tool call  ] {tc['name']}({str(tc['args'])[:80]})")
                any_tools = True
    if not any_tools:
        print("  (no tool calls — skill body stayed on disk)")


show_tool_trace(result_plain, "Control run ('What is 2 + 2?')")
print()
show_tool_trace(result, "Haiku run (skill triggered)")


=== Control run ('What is 2 + 2?') ===
  (no tool calls — skill body stayed on disk)

=== Haiku run (skill triggered) ===
  [tool call  ] read_file({'file_path': '/skills/haiku-report/SKILL.md', 'limit': 1000})
  [tool result] @@ lines 1-11 of 11 @@
---
name: haiku-report
description: Write a status report entirely 


## 2. Memory: AGENTS.md files, always in context

Where skills are *on demand*, **memory** is *always loaded*: `memory=[...]` names [AGENTS.md](https://agents.md/)-style files whose content is injected into the system prompt at startup. Use it for standing rules — persona, conventions, project facts — that should apply on every turn without a tool call.


In [5]:
AGENTS_MD = """# Project context

- The user's name is Sam. Address them by name.
- Sam's reports must always end with the line: "-- filed under project HELIOS".
- Prefer metric units.
"""

remembering_agent = create_deep_agent(
    model=model,
    system_prompt="You are a concise assistant.",
    memory=["/memory/AGENTS.md"],
)

result = remembering_agent.invoke({
    "messages": [{"role": "user", "content": "Summarize in two sentences: the probe travelled 3 miles and found water ice."}],
    "files": {"/memory/AGENTS.md": create_file_data(AGENTS_MD)},
})
print(result["messages"][-1].content)
print()
show_tool_trace(result, "Memory run (notice: 0 read_file calls)")


Sam, the probe traveled approximately 4.8 km (3 miles). It found deposits of water ice at the site.

-- filed under project HELIOS

=== Memory run (notice: 0 read_file calls) ===
  (no tool calls — skill body stayed on disk)


The reply addresses Sam, converts to kilometres, and carries the HELIOS filing line — with **zero `read_file` calls**, because `memory=` injected `AGENTS.md` straight into the system prompt before the first model call. That is memory as *standing context*, versus the skill above, which cost nothing until invoked.

(For files the agent *writes and updates across runs*, pair a `/memories/` route with `FilesystemBackend` as in notebook 01, or `StoreBackend` as in the PRDA app in `src/`.)


## 3. Permissions: the policy gate, one declaration per rule

`permissions=` is declarative path policy for the built-in filesystem tools: pass `FilesystemPermission` rules — **first match wins**, each saying `allow`, `deny`, or `interrupt` (pause for human approval, like notebook 01's HITL). Subagents inherit the rules.

Note: `permissions=` enforces path rules on the **built-in filesystem tools** (`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`), not on custom `@tool` functions.


In [6]:
from deepagents import FilesystemPermission

guarded_agent = create_deep_agent(
    model=model,
    system_prompt="You are a file clerk. Do what the user asks with your file tools.",
    permissions=[
        # Rule 1 (narrower, evaluated first): writes under /archive/drafts/ ARE allowed
        FilesystemPermission(operations=["write"], paths=["/archive/drafts/**"], mode="allow"),
        # Rule 2 (catch-all under /archive/): any other write under /archive/ is denied
        FilesystemPermission(operations=["write"], paths=["/archive/**"], mode="deny"),
    ],
)

result = guarded_agent.invoke({
    "messages": [{"role": "user", "content": (
        "1. Read /archive/q3.txt and tell me what it says. "
        "2. Try to update /archive/q3.txt to say 'Q3 is closed.' "
        "3. Write that note to /archive/drafts/q3_note.txt instead."
    )}],
    "files": {"/archive/q3.txt": create_file_data("Q3 is still open.")},
})
print(result["messages"][-1].content)
print()
show_tool_trace(result, "Permissions trace (read allowed, /archive/q3.txt denied, /archive/drafts/ allowed)")


Done — here are the results of each step:

1) I read /archive/q3.txt. It contains:
Q3 is still open.

2) I attempted to update /archive/q3.txt to say "Q3 is closed." but was unable to: the edit failed with "permission denied for write on /archive/q3.txt".

3) I created /archive/drafts/q3_note.txt and wrote the note there with the content:
Q3 is closed.

=== Permissions trace (read allowed, /archive/q3.txt denied, /archive/drafts/ allowed) ===
  [tool call  ] ls({'path': '/archive'})
  [tool result] ['/archive/q3.txt']
  [tool call  ] read_file({'file_path': '/archive/q3.txt'})
  [tool result] @@ lines 1-1 of 1 @@
Q3 is still open.
  [tool call  ] edit_file({'file_path': '/archive/q3.txt', 'old_string': 'Q3 is still open.', 'new_string')
  [tool result] Error: permission denied for write on /archive/q3.txt
  [tool call  ] write_file({'file_path': '/archive/drafts/q3_note.txt', 'content': 'Q3 is closed.'})
  [tool result] Updated file /archive/drafts/q3_note.txt


Look at the tool trace above:
1. `read_file('/archive/q3.txt')` succeeds (only `write` was restricted).
2. Editing `/archive/q3.txt` hits Rule 2 and comes back as a permission-denied **`ToolMessage`** — a refusal as information the model can react to, not a crash.
3. Writing `/archive/drafts/q3_note.txt` matches Rule 1 first and succeeds (**first match wins**).

Swap `mode="deny"` for `mode="interrupt"` (plus a checkpointer) and the same rule becomes a human-approval gate.


## 4. Retrying failed tools: resilience as middleware

Real tools fail — rate limits, flaky networks. `ToolRetryMiddleware` re-runs a **raised** tool error with exponential backoff before the model ever sees it. (Malformed tool *calls* are a different failure class — deepagents already repairs those in its base stack with `PatchToolCallsMiddleware`.)


In [7]:
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.tools import tool

attempts = {"n": 0}


@tool
def flaky_lookup(city: str) -> str:
    """Look up the population of a city (unreliable network).

    Required for every population question. Pass the city name exactly as given.
    """
    attempts["n"] += 1
    if attempts["n"] < 3:
        raise ConnectionError(f"network glitch on attempt {attempts['n']}")
    return f"{city}: 8.8 million (attempt {attempts['n']} succeeded)"


resilient_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You do not know any populations. "
        "Call flaky_lookup and answer from its result. "
        "Do not ask clarifying questions."
    ),
    tools=[flaky_lookup],
    middleware=[ToolRetryMiddleware(max_retries=4, retry_on=(ConnectionError,))],
)

attempts["n"] = 0  # reset before invoke so re-running this cell always triggers 2 glitches + 1 success
result = resilient_agent.invoke(
    {"messages": [{"role": "user", "content":
        "What is the population of London, United Kingdom?"}]}
)
print(result["messages"][-1].content)
print("tool attempts:", attempts["n"])


The population of London, United Kingdom is about 8.8 million (result from the lookup).
tool attempts: 3


Three attempts, one tool call from the model's point of view: the first two raises never reached the conversation. Siblings worth knowing from the same middleware family: `ModelRetryMiddleware` and `ModelFallbackMiddleware` (retry or switch models on provider errors).


## 5. Structured output: typed responses with `response_format=`

When an agent sits behind an API or feeds a downstream pipeline, free-form prose is brittle. Pass a Pydantic model (or JSON schema) to `response_format=` on `create_deep_agent` — or on an individual `SubAgent` — and the agent still uses its filesystem and tools normally, then returns a validated object under `result["structured_response"]`.


In [8]:
from pydantic import BaseModel, Field


class QuarterReview(BaseModel):
    """Structured summary of a quarterly archive file."""

    quarter: str = Field(description="Quarter identifier, e.g. Q3")
    is_closed: bool = Field(description="Whether the quarter is marked closed")
    next_action: str = Field(description="One-sentence recommended next step")


typed_agent = create_deep_agent(
    model=model,
    system_prompt="Read the requested file and return the structured review.",
    response_format=QuarterReview,
)

result = typed_agent.invoke({
    "messages": [{"role": "user", "content": "Review /archive/q3.txt."}],
    "files": {"/archive/q3.txt": create_file_data("Q3 is still open; awaiting sign-off from finance.")},
})

review: QuarterReview = result["structured_response"]
print("Type:       ", type(review).__name__)
print("Quarter:    ", review.quarter)
print("Is closed?: ", review.is_closed)
print("Next action:", review.next_action)


Type:        QuarterReview
Quarter:     Q3
Is closed?:  False
Next action: Follow up with finance to obtain sign-off so Q3 can be closed.


## 6. Wrap-up: Which context or policy knob when?

Across Notebooks 01 and 02, you have three ways to give a Deep Agent knowledge and two ways to gate its actions:

| Mechanism | When it loads / runs | Best for |
|---|---|---|
| `memory=["/memory/AGENTS.md"]` | Injected into system prompt on **every turn** (0 tool calls) | Short standing rules, user preferences, project conventions |
| `skills=["/skills/"]` | Frontmatter catalog at start; full `SKILL.md` read **on demand** | Reusable playbooks, templates, domain workflows |
| `/memories/` via `Backend` (`Filesystem` / `Store`) | Only when the agent calls `read_file` / `write_file` | Durable notes and artifacts saved across threads or runs |
| `permissions=[FilesystemPermission(...)]` | Checked before built-in file tools (`allow` / `deny` / `interrupt`) | Path-based sandboxing and read-only folders |
| `interrupt_on={"tool": True}` | Pauses graph before named tools (Notebook 01) | Human approval on custom actions (`send_email`, `delete_file`, …) |

> A framework gives you an agent loop. A **harness** gives the loop its memory, skills, policies, failure handling, and typed outputs — and the mark of a good one is how little code each of those costs you.

The PRDA application in `src/deepagents_basics/` is the capstone: Notebook 01's orchestration, planning, streaming, and `/memories/` store assembled into a product team of six subagents.
